# Train MatGPT Base Models on a Colab T4

This notebook runs the base-pretraining pipeline for the course validation models:

- `MatGPT-Mini 8M` on TinyStories
- `MatGPT-Tiny 59M` on BabyLM-2026-Strict

Before running: choose **Runtime -> Change runtime type -> T4 GPU**. Active data stays on fast, ephemeral `/content`; prepared artifacts, run evidence, and checkpoints persist in Google Drive.

## 1. Choose one stage

Run `prepare`, then stop and review its persisted `preflight.json` and `benchmark.json` before selecting `smoke`. Continue with `pilot` and then `evaluate`. Reaching pilot step 306 is not approval: review the persisted evidence with Codex before manually selecting `full`. Select `evaluate` again after the full run.

In [ ]:
# @title Run settings
MODEL = "mini_8m"  # @param ["mini_8m", "tiny_59m"]
RUN_STAGE = "prepare"  # @param ["prepare", "smoke", "pilot", "full", "evaluate"]
SMOKE_MAX_STEPS = 20
RESUME_CHECK_STEPS = 5
PILOT_STOP_STEP = 306
FORCE_REBUILD_PREPARED = False  # @param {type:"boolean"}

# If you publish this project to GitHub, paste the repo URL here.
# If left blank, the notebook expects PROJECT_DIR to already contain this repo.
REPO_URL = ""  # @param {type:"string"}
PROJECT_DIR = "/content/train-llm-from-scratch"  # @param {type:"string"}

# W&B is optional but recommended for a serious run because it preserves curves
# even if the Colab tab disconnects.
ENABLE_WANDB = True  # @param {type:"boolean"}
WANDB_PROJECT = "matgpt-t4-base"  # @param {type:"string"}
WANDB_ENTITY = ""  # @param {type:"string"}
assert MODEL in {"mini_8m", "tiny_59m"}
assert RUN_STAGE in {"prepare", "smoke", "pilot", "full", "evaluate"}
assert not FORCE_REBUILD_PREPARED or RUN_STAGE == "prepare", (
    "FORCE_REBUILD_PREPARED may only be enabled for the prepare stage."
)

## 2. Mount Google Drive

Colab VMs disappear. Drive stores durable checkpoints, metrics, preflight and benchmark JSON, plus synchronized copies of prepared artifacts.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Locate or clone the project

This cell keeps the project checkout on local `/content`. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` with repo read access. If `REPO_URL` is blank, `PROJECT_DIR` must already contain the repository.

In [ ]:
import os
import subprocess
from pathlib import Path
from urllib.parse import urlparse, urlunparse

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

def github_clone_url(repo_url: str, token: str | None) -> str:
    """Attach a GitHub token for private HTTPS clones without changing REPO_URL."""
    if not token or not repo_url.startswith("https://github.com/"):
        return repo_url
    parsed = urlparse(repo_url)
    return urlunparse(parsed._replace(netloc=f"x-access-token:{token}@{parsed.netloc}"))

def git_run(args, label: str, token: str | None = None):
    """Run git and show stdout/stderr. This avoids opaque CalledProcessError traces."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(token, "<GITHUB_TOKEN>") if token else result.stdout
    stderr = result.stderr.replace(token, "<GITHUB_TOKEN>") if token else result.stderr
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} with exit code {result.returncode}.\n\n"
            f"stdout:\n{stdout}\n\n"
            f"stderr:\n{stderr}\n\n"
            "If this is a private GitHub repo, add a Colab secret named GITHUB_TOKEN "
            "with repo read access, then rerun this cell."
        )
    if stdout:
        print(stdout)
    if stderr:
        print(stderr)
    return result

project_dir = Path(PROJECT_DIR)
github_token = get_colab_secret("GITHUB_TOKEN")

if project_dir.exists() and (project_dir / ".git").exists():
    print("Existing git checkout found. Pulling latest changes.")
    git_pull = git_run(["git", "-C", str(project_dir), "pull", "--ff-only"], "git pull", github_token)
elif REPO_URL:
    if project_dir.exists() and any(project_dir.iterdir()):
        raise RuntimeError(
            f"PROJECT_DIR exists but is not a git checkout: {project_dir}\n"
            "Choose a different PROJECT_DIR, delete the partial folder, or copy the repo there manually."
        )
    project_dir.parent.mkdir(parents=True, exist_ok=True)
    clone_url = github_clone_url(REPO_URL, github_token)
    git_run(["git", "clone", clone_url, str(project_dir)], "Git clone failed", github_token)
elif not project_dir.exists():
    raise RuntimeError(
        "REPO_URL is blank and PROJECT_DIR does not exist. Set REPO_URL to the GitHub repo, "
        "or place the project checkout at PROJECT_DIR first."
    )

os.chdir(project_dir)
print("Current directory:", Path.cwd())
assert Path("scripts/pretrain.py").exists(), "PROJECT_DIR must point at the MatGPT repository root."

## 4. Install dependencies

This installs the local package, dataset tools, W&B, and optional Colab helpers such as bitsandbytes.

In [ ]:
%pip install -q -e ".[test,colab]" huggingface_hub wandb

## 5. Authenticate Hugging Face and W&B

Recommended Colab secrets:

- `HF_TOKEN`: Hugging Face token for dataset access
- `WANDB_API_KEY`: Weights & Biases API key

If a secret is missing, the notebook falls back to the normal interactive login prompt.

In [ ]:
from huggingface_hub import login as hf_login
import wandb

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

hf_token = get_colab_secret("HF_TOKEN")
if hf_token:
    hf_login(token=hf_token, add_to_git_credential=False)
else:
    hf_login(add_to_git_credential=False)

if ENABLE_WANDB:
    wandb_key = get_colab_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
    else:
        wandb.login()
else:
    print("W&B disabled for this run.")

## 6. Gate storage and the GPU

Every stage prints local and Drive capacity before expensive work. Preparation requires at least 20 GiB free on `/content`; the `prepare`, `smoke`, `pilot`, and `full` evidence gates require CUDA on an NVIDIA T4.

In [ ]:
!nvidia-smi

import shutil
import torch

LOCAL_MIN_FREE_GIB = 20
local_usage = shutil.disk_usage("/content")
drive_usage = shutil.disk_usage("/content/drive/MyDrive")
print("/content disk usage:", local_usage)
print("Drive disk usage:", drive_usage)
assert local_usage.free >= LOCAL_MIN_FREE_GIB * 1024**3, (
    f"Need at least {LOCAL_MIN_FREE_GIB} GiB free on /content; "
    f"observed {local_usage.free / 1024**3:.2f} GiB."
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable"
print("gpu:", gpu_name)
if RUN_STAGE in {"prepare", "smoke", "pilot", "full"}:
    assert torch.cuda.is_available(), f"{RUN_STAGE} requires CUDA."
    assert "T4" in gpu_name, f"{RUN_STAGE} requires an NVIDIA T4; observed {gpu_name!r}."

## 7. Build the fixed-path Colab config

All stages use the unchanged full training schedule. Normalized data, tokenizer files, and shards work from a stable local path and are restored from or synchronized to Drive; run evidence writes directly to Drive.

In [ ]:
import hashlib
import json
import shutil
import uuid
import yaml
from pathlib import Path

base_config_path = Path("configs/matgpt_mini_8m.yaml") if MODEL == "mini_8m" else Path("configs/matgpt_tiny_59m.yaml")
cfg = yaml.safe_load(base_config_path.read_text())

LOCAL_ROOT = Path("/content/matgpt_work") / cfg["run"]["name"]
DRIVE_ROOT = Path("/content/drive/MyDrive/matgpt_artifacts") / cfg["run"]["name"]
cfg["run"]["output_dir"] = str(DRIVE_ROOT / "run")
cfg["dataset"]["normalized_dir"] = str(LOCAL_ROOT / "normalized")
cfg["tokenizer"]["output_dir"] = str(LOCAL_ROOT / "tokenizer")
cfg["sharding"]["output_dir"] = str(LOCAL_ROOT / "shards")

cfg["tracking"]["wandb"]["enabled"] = bool(ENABLE_WANDB)
cfg["tracking"]["wandb"]["project"] = WANDB_PROJECT
cfg["tracking"]["wandb"]["entity"] = WANDB_ENTITY or None

PERSISTED_ARTIFACT_DIRS = ("normalized", "tokenizer", "shards")


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _sha256_json(payload: dict) -> str:
    encoded = json.dumps(
        payload, sort_keys=True, ensure_ascii=False, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def _read_json_object(path: Path, label: str) -> dict:
    if not path.is_file():
        raise ValueError(f"Missing {label}: {path}")
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid {label} JSON: {path}") from error
    if not isinstance(payload, dict):
        raise ValueError(f"{label} must contain a JSON object: {path}")
    return payload


def _validate_payload_hash(payload: dict, field: str, label: str) -> None:
    stored = payload.get(field)
    hash_payload = dict(payload)
    hash_payload.pop(field, None)
    if not isinstance(stored, str) or stored != _sha256_json(hash_payload):
        raise ValueError(f"Invalid {label} {field}")


def validate_normalized_artifacts(
    root: Path, *, splits: tuple[str, ...], revision: str
) -> None:
    root = Path(root)
    manifest = _read_json_object(root / "manifest.json", "dataset manifest")
    _validate_payload_hash(manifest, "manifest_sha256", "dataset manifest")
    if manifest.get("version_or_commit") != revision:
        raise ValueError("Dataset manifest revision does not match the config")
    split_stats = manifest.get("split_stats")
    if not isinstance(split_stats, dict):
        raise ValueError("Dataset manifest is missing split_stats")
    for split in splits:
        path = root / f"{split}.jsonl"
        if not path.is_file():
            raise ValueError(f"Missing normalized split: {path}")
        row_count = 0
        with path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f"Invalid normalized JSONL at {path}:{line_number}") from error
                if not isinstance(row, dict) or not isinstance(row.get("text"), str):
                    raise ValueError(f"Invalid normalized row at {path}:{line_number}")
                row_count += 1
        expected_count = split_stats.get(split, {}).get("document_count")
        if row_count < 1 or row_count != expected_count:
            raise ValueError(
                f"Normalized split count mismatch for {split}: "
                f"observed={row_count} expected={expected_count}"
            )


def validate_tokenizer_artifacts(root: Path) -> None:
    root = Path(root)
    tokenizer_path = root / "tokenizer.json"
    _read_json_object(tokenizer_path, "tokenizer")
    metadata = _read_json_object(root / "special_tokens.json", "tokenizer metadata")
    observed_hash = metadata.get("tokenizer_sha256")
    if not isinstance(observed_hash, str) or observed_hash != _sha256_file(tokenizer_path):
        raise ValueError("Tokenizer metadata hash does not match tokenizer.json")


def validate_shard_artifacts(
    root: Path,
    *,
    metadata_root: Path,
    splits: tuple[str, ...],
    tokenizer_sha256: str,
) -> None:
    root = Path(root)
    metadata_root = Path(metadata_root).resolve()
    dtype_sizes = {"uint16": 2, "uint32": 4}
    split_payloads = {}
    for split in splits:
        metadata_path = root / f"{split}_metadata.json"
        metadata = _read_json_object(metadata_path, f"{split} shard metadata")
        _validate_payload_hash(metadata, "metadata_sha256", f"{split} shard metadata")
        if metadata.get("split") != split:
            raise ValueError(f"Shard metadata split mismatch for {split}")
        if metadata.get("tokenizer_sha256") != tokenizer_sha256:
            raise ValueError(f"Shard tokenizer hash mismatch for {split}")
        dtype_size = dtype_sizes.get(metadata.get("dtype"))
        if dtype_size is None:
            raise ValueError(f"Unsupported shard dtype for {split}")
        shards = metadata.get("shards")
        if not isinstance(shards, list) or not shards:
            raise ValueError(f"Shard metadata contains no payloads for {split}")
        total_tokens = 0
        for shard in shards:
            if not isinstance(shard, dict):
                raise ValueError(f"Invalid shard entry for {split}")
            try:
                relative_path = Path(shard["path"]).resolve().relative_to(metadata_root)
            except (KeyError, ValueError) as error:
                raise ValueError(f"Shard path escapes the fixed local root for {split}") from error
            payload_path = root / relative_path
            if not payload_path.is_file():
                raise ValueError(f"Referenced shard payload is missing: {payload_path}")
            num_tokens = int(shard.get("num_tokens", -1))
            if num_tokens < 1 or payload_path.stat().st_size != num_tokens * dtype_size:
                raise ValueError(f"Shard payload size mismatch: {payload_path}")
            if shard.get("sha256") != _sha256_file(payload_path):
                raise ValueError(f"Shard payload hash mismatch: {payload_path}")
            total_tokens += num_tokens
        if total_tokens != int(metadata.get("total_tokens", -1)):
            raise ValueError(f"Shard token total mismatch for {split}")
        split_payloads[split] = metadata
    combined = _read_json_object(root / "metadata.json", "combined shard metadata")
    _validate_payload_hash(combined, "metadata_sha256", "combined shard metadata")
    if combined.get("splits") != split_payloads:
        raise ValueError("Combined shard metadata does not match split metadata")


def validate_artifact_directory(name: str, path: Path) -> None:
    validation_split = (
        cfg["dataset"].get("validation_split")
        or cfg["dataset"].get("generated_validation_split")
    )
    splits = (cfg["dataset"]["train_split"], validation_split)
    if name == "normalized":
        validate_normalized_artifacts(
            path, splits=splits, revision=cfg["dataset"]["revision"]
        )
    elif name == "tokenizer":
        validate_tokenizer_artifacts(path)
    elif name == "shards":
        tokenizer_metadata = _read_json_object(
            LOCAL_ROOT / "tokenizer" / "special_tokens.json",
            "local tokenizer metadata",
        )
        validate_shard_artifacts(
            path,
            metadata_root=LOCAL_ROOT / "shards",
            splits=splits,
            tokenizer_sha256=tokenizer_metadata["tokenizer_sha256"],
        )
    else:
        raise ValueError(f"Unknown prepared artifact directory: {name}")


def artifact_status(name: str, path: Path) -> tuple[bool, str]:
    try:
        validate_artifact_directory(name, path)
    except Exception as error:
        return False, str(error)
    return True, "complete"


def _remove_path(path: Path) -> None:
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()


def _replace_directory_from_snapshot(snapshot: Path, destination: Path) -> None:
    backup = destination.parent / f".{destination.name}.backup-{uuid.uuid4().hex}"
    try:
        if destination.exists():
            destination.replace(backup)
        snapshot.replace(destination)
    except Exception:
        if not destination.exists() and backup.exists():
            backup.replace(destination)
        raise
    finally:
        _remove_path(snapshot)
        _remove_path(backup)


def publish_directory_snapshot(source: Path, destination: Path, validate_snapshot) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    snapshot = destination.parent / f".{destination.name}.syncing-{uuid.uuid4().hex}"
    try:
        shutil.copytree(source, snapshot)
        validate_snapshot(snapshot)
        _replace_directory_from_snapshot(snapshot, destination)
    finally:
        _remove_path(snapshot)


def restore_directory_snapshot(source: Path, destination: Path, validate_snapshot) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    snapshot = destination.parent / f".{destination.name}.restoring-{uuid.uuid4().hex}"
    try:
        shutil.copytree(source, snapshot)
        validate_snapshot(snapshot)
        _replace_directory_from_snapshot(snapshot, destination)
    finally:
        _remove_path(snapshot)


def remove_ephemeral_artifact(name: str) -> None:
    if name not in PERSISTED_ARTIFACT_DIRS:
        raise ValueError(f"Refusing to remove unknown artifact {name!r}")
    local_path = (LOCAL_ROOT / name).resolve()
    local_path.relative_to(LOCAL_ROOT.resolve())
    _remove_path(local_path)


def restore_artifacts_from_drive():
    restored = []
    rebuild = []
    for name in PERSISTED_ARTIFACT_DIRS:
        local_path = LOCAL_ROOT / name
        drive_path = DRIVE_ROOT / name
        local_complete, local_error = artifact_status(name, local_path)
        if local_complete:
            continue
        drive_complete, drive_error = artifact_status(name, drive_path)
        if drive_complete:
            restore_directory_snapshot(
                drive_path,
                local_path,
                lambda snapshot, artifact_name=name: validate_artifact_directory(
                    artifact_name, snapshot
                ),
            )
            restored.append(name)
        elif local_path.exists() or drive_path.exists():
            if not FORCE_REBUILD_PREPARED:
                raise RuntimeError(
                    f"Prepared artifact {name!r} is incomplete locally and no complete "
                    f"Drive snapshot is available. local={local_error}; drive={drive_error}. "
                    "Set FORCE_REBUILD_PREPARED=True to remove only the ephemeral local "
                    "copy and rebuild during the prepare stage."
                )
            remove_ephemeral_artifact(name)
            rebuild.append(name)
    print("Restored from complete Drive snapshots:", restored or "nothing")
    print("Marked for local rebuild:", rebuild or "nothing")


def validate_all_prepared_artifacts() -> None:
    for name in PERSISTED_ARTIFACT_DIRS:
        validate_artifact_directory(name, LOCAL_ROOT / name)


def sync_artifacts_to_drive():
    synchronized = []
    for name in PERSISTED_ARTIFACT_DIRS:
        local_path = LOCAL_ROOT / name
        validate_artifact_directory(name, local_path)
        publish_directory_snapshot(
            local_path,
            DRIVE_ROOT / name,
            lambda snapshot, artifact_name=name: validate_artifact_directory(
                artifact_name, snapshot
            ),
        )
        synchronized.append(name)
    print("Published complete Drive snapshots:", synchronized)


LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
restore_artifacts_from_drive()

colab_config_dir = LOCAL_ROOT / "config"
colab_config_dir.mkdir(parents=True, exist_ok=True)
colab_config_path = colab_config_dir / f"{MODEL}.yaml"
colab_config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

print("Local working root:", LOCAL_ROOT)
print("Persistent Drive root:", DRIVE_ROOT)
print("Persistent run directory:", cfg["run"]["output_dir"])
print("Local normalized data:", cfg["dataset"]["normalized_dir"])
print("Local tokenizer:", cfg["tokenizer"]["output_dir"])
print("Local shards:", cfg["sharding"]["output_dir"])
print("Wrote config:", colab_config_path)
print(json.dumps({
    "model": MODEL,
    "run_stage": RUN_STAGE,
    "max_tokens": cfg["training"]["max_tokens"],
    "wandb_enabled": cfg["tracking"]["wandb"]["enabled"],
}, indent=2))

## 8. Prepare once, then synchronize

The `prepare` stage skips only hash- and payload-complete artifacts. Incomplete local data restores from a complete Drive snapshot; otherwise set `FORCE_REBUILD_PREPARED=True` to remove only the ephemeral local copy and rebuild it. Validated artifacts publish to Drive through temporary replacement snapshots, never directory merges.

In [ ]:
import shlex
import subprocess


def run_command(command):
    """Run a command, expose its evidence, and fail with full diagnostics."""
    printable = " ".join(shlex.quote(str(part)) for part in command)
    print("\n$", printable)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {printable}\n\n"
            f"stdout:\n{result.stdout}\n\n"
            f"stderr:\n{result.stderr}"
        )
    return result


validation_split = (
    cfg["dataset"].get("validation_split")
    or cfg["dataset"].get("generated_validation_split")
)
preparation_steps = [
    (
        "normalized",
        "normalized dataset",
        ["python", "scripts/prepare_dataset.py", "--config", str(colab_config_path)],
    ),
    (
        "tokenizer",
        "tokenizer",
        ["python", "scripts/train_tokenizer.py", "--config", str(colab_config_path)],
    ),
    (
        "shards",
        "token shards",
        ["python", "scripts/tokenize_and_shard.py", "--config", str(colab_config_path)],
    ),
]

def prepare_artifacts_for_stage(
    *,
    run_stage,
    preparation_steps,
    local_root,
    artifact_status_fn,
    run_command_fn,
    validate_artifact_fn,
    validate_all_fn,
    sync_fn,
    emit=print,
):
    if run_stage != "prepare":
        emit(f"Preparation commands skipped for stage {run_stage!r}.")
        return
    for artifact_name, label, command in preparation_steps:
        artifact_path = local_root / artifact_name
        complete, error = artifact_status_fn(artifact_name, artifact_path)
        if complete:
            emit(f"Skipping {label}; integrity validation passed.")
        else:
            emit(f"Rebuilding {label}: {error}")
            run_command_fn(command)
            validate_artifact_fn(artifact_name, artifact_path)
    validate_all_fn()
    sync_fn()


prepare_artifacts_for_stage(
    run_stage=RUN_STAGE,
    preparation_steps=preparation_steps,
    local_root=LOCAL_ROOT,
    artifact_status_fn=artifact_status,
    run_command_fn=run_command,
    validate_artifact_fn=validate_artifact_directory,
    validate_all_fn=validate_all_prepared_artifacts,
    sync_fn=sync_artifacts_to_drive,
)

## 9. Gate training with preflight and benchmark evidence

The `prepare` stage and every training stage run strict T4 preflight and a five-step, temporary batch benchmark. Review the persisted `preflight.json` and `benchmark.json`; pretraining is blocked unless the configured batch has finite loss and pre-clip gradient norm, positive throughput, and memory fraction below 0.90. The benchmark updates only its temporary model and creates no training checkpoint.

In [ ]:
import json
import math
from pathlib import Path


def run_evidence_gate(
    *,
    run_stage,
    config_path,
    run_output_dir,
    batch_sizes,
    micro_batch_size,
    run_command_fn,
    emit=print,
):
    evidence_stages = {"prepare", "smoke", "pilot", "full"}
    if run_stage not in evidence_stages:
        emit(f"Preflight and benchmark evidence gates skipped for {run_stage!r}.")
        return None

    run_command_fn([
        "python", "scripts/preflight_t4.py",
        "--config", str(config_path),
        "--require-t4",
        "--min-free-disk-gb", "20",
    ])
    preflight_path = Path(run_output_dir) / "preflight.json"
    preflight_report = json.loads(preflight_path.read_text(encoding="utf-8"))
    emit(f"Persisted preflight evidence: {preflight_path}")
    emit(json.dumps(preflight_report, indent=2, sort_keys=True))
    assert preflight_report["status"] == "pass", "Preflight gate did not pass."

    benchmark = run_command_fn([
        "python", "scripts/benchmark_t4.py",
        "--config", str(config_path),
        "--batch-sizes", batch_sizes,
        "--steps", "5",
    ])
    benchmark_path = Path(run_output_dir) / "benchmark.json"
    benchmark_path.parent.mkdir(parents=True, exist_ok=True)
    benchmark_path.write_text(benchmark.stdout, encoding="utf-8")
    benchmark_report = json.loads(benchmark.stdout)
    selected_batch = next(
        result
        for result in benchmark_report["results"]
        if result["batch_size"] == micro_batch_size
    )
    emit(f"Persisted benchmark evidence: {benchmark_path}")
    emit(json.dumps({
        "batch_size": selected_batch["batch_size"],
        "loss": selected_batch.get("loss"),
        "grad_norm": selected_batch.get("grad_norm"),
        "memory_fraction": selected_batch.get("memory_fraction"),
        "tokens_per_second": selected_batch.get("tokens_per_second"),
    }, indent=2, sort_keys=True))
    if selected_batch["status"] != "ok":
        raise RuntimeError(f"Configured batch benchmark failed: {selected_batch}")
    for field in ("loss", "grad_norm", "tokens_per_second", "memory_fraction"):
        if not math.isfinite(float(selected_batch[field])):
            raise FloatingPointError(f"Benchmark {field} is not finite: {selected_batch[field]}")
    if selected_batch["tokens_per_second"] <= 0:
        raise RuntimeError("Benchmark throughput must be positive.")
    if selected_batch["memory_fraction"] >= 0.90:
        raise RuntimeError(f"Benchmark memory fraction is too high: {selected_batch['memory_fraction']:.3f}")
    if run_stage == "prepare":
        emit("Prepare gate complete. Stop here and review preflight.json and benchmark.json before selecting smoke.")
    return {
        "preflight_path": preflight_path,
        "benchmark_path": benchmark_path,
        "selected_batch": selected_batch,
    }


batch_sizes = "8,16,24,32" if MODEL == "mini_8m" else "2,4,6,8"
gate_evidence = run_evidence_gate(
    run_stage=RUN_STAGE,
    config_path=colab_config_path,
    run_output_dir=cfg["run"]["output_dir"],
    batch_sizes=batch_sizes,
    micro_batch_size=cfg["training"]["micro_batch_size"],
    run_command_fn=run_command,
)

## 10. Run the selected training stage

`--max-steps` always means additional successful optimizer updates; it never rewrites `max_tokens` or the full learning-rate schedule (6,104 steps for the 200M-token Mini run). Smoke recovers only from no checkpoint, exact step 20, or exact step 25; every other smoke lineage is rejected. Pilot stops at global step 306. Reaching 306 is not approval: the user and Codex must review the evidence before the operator manually selects `full`.

In [ ]:
from matgpt.training.schedule import build_training_schedule


def checkpoint_step(path) -> int:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    return int(payload["state"]["global_step"])


def assert_checkpoint_step(path, expected_step: int) -> int:
    observed_step = checkpoint_step(path)
    assert observed_step == expected_step, (
        f"Checkpoint step mismatch: observed={observed_step} expected={expected_step}"
    )
    return observed_step


def smoke_actions_for_step(current_step: int | None) -> tuple[str, ...]:
    if current_step is None:
        return ("initial", "resume_check")
    if current_step == 20:
        return ("resume_check",)
    if current_step == 25:
        return ()
    raise AssertionError(
        f"Unexpected smoke checkpoint lineage: global_step={current_step}; "
        "expected no checkpoint, step 20, or step 25."
    )


base_train_cmd = ["python", "scripts/pretrain.py", "--config", str(colab_config_path)]
latest = Path(cfg["run"]["output_dir"]) / "checkpoints" / "latest.pt"

if RUN_STAGE == "smoke":
    current_step = checkpoint_step(latest) if latest.exists() else None
    for action in smoke_actions_for_step(current_step):
        if action == "initial":
            run_command(base_train_cmd + ["--max-steps", str(SMOKE_MAX_STEPS)])
            assert_checkpoint_step(latest, 20)
        elif action == "resume_check":
            assert_checkpoint_step(latest, 20)
            run_command(
                base_train_cmd
                + ["--resume-from", str(latest), "--max-steps", str(RESUME_CHECK_STEPS)]
            )
            assert_checkpoint_step(latest, 25)
    print("Smoke and resume check are complete at global step 25.")
elif RUN_STAGE == "pilot":
    assert latest.exists(), "Run the smoke and resume check first."
    current_step = checkpoint_step(latest)
    remaining = PILOT_STOP_STEP - current_step
    assert remaining > 0, f"Pilot already reached: global_step={current_step}"
    run_command(base_train_cmd + ["--resume-from", str(latest), "--max-steps", str(remaining)])
    assert_checkpoint_step(latest, 306)
    print("Pilot reached step 306. Select evaluate; do not select full yet.")
elif RUN_STAGE == "full":
    assert latest.exists(), "Run and approve the pilot first."
    current_step = checkpoint_step(latest)
    assert current_step >= PILOT_STOP_STEP, "Pilot gate has not been reached."
    full_stop_step = build_training_schedule(cfg).total_steps
    run_command(base_train_cmd + ["--resume-from", str(latest)])
    assert_checkpoint_step(latest, full_stop_step)
else:
    print(f"No training command runs during stage {RUN_STAGE!r}.")

## 11. Evaluate every available review checkpoint

The `evaluate` stage requires and evaluates both `latest.pt` and `best.pt`, verifies that the complete `latest.pt` resume state loads without taking an update, then writes the durable run summary. Run this after pilot and again after full training.

In [ ]:
import json

run_dir = Path(cfg["run"]["output_dir"])
checkpoint_dir = run_dir / "checkpoints"
best = checkpoint_dir / "best.pt"
latest = checkpoint_dir / "latest.pt"

if RUN_STAGE == "evaluate":
    assert latest.is_file(), f"Missing required latest.pt: {latest}"
    assert best.is_file(), f"Missing required best.pt: {best}"
    evaluation_checkpoints = (latest, best)
    for checkpoint in evaluation_checkpoints:
        run_command([
            "python", "scripts/evaluate.py",
            "--config", str(colab_config_path),
            "--checkpoint", str(checkpoint),
        ])
    resume_verification = run_command([
        "python", "scripts/pretrain.py",
        "--config", str(colab_config_path),
        "--resume-from", str(latest),
        "--verify-only",
    ])
    resume_report = json.loads(resume_verification.stdout)
    assert resume_report["resume_verified"] is True
    resume_report_path = run_dir / "resume_verification.json"
    resume_report_path.write_text(
        json.dumps(resume_report, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    print("Persisted resume verification evidence:", resume_report_path)
    run_command(["python", "scripts/summarize_run.py", "--run-dir", str(run_dir)])
else:
    print(f"Evaluation commands skipped for stage {RUN_STAGE!r}.")

## 12. Review the persisted gate evidence

This review is intentionally manual. It displays the summary, last 30 metric rows, losses, skipped updates, memory behavior, benchmark fraction, evaluation files, and latest sample. The calculations never promote the notebook to `full`.

In [ ]:
import math

import pandas as pd
from IPython.display import Markdown, display

if RUN_STAGE == "evaluate":
    summary_path = run_dir / "run_summary.md"
    assert summary_path.is_file(), f"Missing run summary: {summary_path}"
    display(Markdown(summary_path.read_text(encoding="utf-8")))

    metrics_path = run_dir / "metrics.csv"
    assert metrics_path.is_file(), f"Missing metrics: {metrics_path}"
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail(30))
    for column in ["train_loss", "val_loss"]:
        finite_rows = metrics.dropna(subset=[column]).copy()
        finite_rows[column] = pd.to_numeric(finite_rows[column], errors="coerce")
        finite_rows = finite_rows[finite_rows[column].map(math.isfinite)]
        finite_rows.plot(x="tokens_processed", y=column, title=column)

    skipped_columns = [
        column for column in (
            "global_step",
            "optimizer_step_skipped",
            "optimizer_steps_skipped_total",
            "consecutive_optimizer_steps_skipped",
        )
        if column in metrics.columns
    ]
    display(metrics.loc[metrics["event"] == "train", skipped_columns].tail(30))

    finite_training_fields = [
        "train_loss", "tokens_per_second", "peak_memory_mb",
        "optimizer_steps_skipped_total",
    ]
    for field in finite_training_fields:
        metrics[field] = pd.to_numeric(metrics[field], errors="coerce")
    train_rows = metrics.dropna(subset=["train_loss"]).copy()
    train_rows = train_rows[
        train_rows[finite_training_fields].apply(
            lambda row: all(math.isfinite(float(value)) for value in row), axis=1
        )
    ]
    window = max(5, math.ceil(len(train_rows) * 0.20))
    assert len(train_rows) >= window * 2, "Not enough logged pilot rows for early/late comparison."
    early = train_rows.head(window)
    late = train_rows.tail(window)
    assert float(early["tokens_per_second"].median()) > 0
    pilot_gate = {
        "early_loss_median": float(early["train_loss"].median()),
        "late_loss_median": float(late["train_loss"].median()),
        "throughput_ratio": float(late["tokens_per_second"].median() / early["tokens_per_second"].median()),
        "maximum_peak_memory_mb": float(train_rows["peak_memory_mb"].max()),
        "skipped_updates_total": int(train_rows["optimizer_steps_skipped_total"].max()),
    }
    pilot_gate["loss_improved"] = pilot_gate["late_loss_median"] < pilot_gate["early_loss_median"]
    pilot_gate["throughput_stable"] = pilot_gate["throughput_ratio"] >= 0.80

    val_rows = metrics.dropna(subset=["val_loss"]).copy()
    val_rows["val_loss"] = pd.to_numeric(val_rows["val_loss"], errors="coerce")
    val_rows = val_rows[val_rows["val_loss"].map(math.isfinite)]
    assert len(val_rows) >= 2, "Need at least two finite validation rows for comparison."
    pilot_gate["first_val_loss"] = float(val_rows.iloc[0]["val_loss"])
    pilot_gate["last_val_loss"] = float(val_rows.iloc[-1]["val_loss"])
    pilot_gate["validation_improved"] = pilot_gate["last_val_loss"] < pilot_gate["first_val_loss"]

    benchmark_path = run_dir / "benchmark.json"
    benchmark_report = json.loads(benchmark_path.read_text(encoding="utf-8"))
    selected_batch = next(
        result for result in benchmark_report["results"]
        if result["batch_size"] == cfg["training"]["micro_batch_size"]
    )
    pilot_gate["benchmark_memory_fraction"] = float(selected_batch["memory_fraction"])
    print(json.dumps(pilot_gate, indent=2, sort_keys=True))

    peak_memory_values = train_rows["peak_memory_mb"].tolist()
    if len(peak_memory_values) > 1 and all(
        current > previous for previous, current in zip(peak_memory_values, peak_memory_values[1:])
    ):
        print("WARNING: every logged peak-memory value is strictly greater than its predecessor.")

    evaluation_files = sorted((run_dir / "evaluation").glob("*.json"))
    print("Evaluation files:", [str(path) for path in evaluation_files])
    sample_files = sorted((run_dir / "samples").glob("samples_*.json"))
    if sample_files:
        print("Latest sample file:", sample_files[-1])
        print(sample_files[-1].read_text(encoding="utf-8")[:4000])
    else:
        print("No periodic sample file exists yet.")
else:
    print("Select evaluate after pilot to review promotion evidence.")

## 13. Generate from the checkpoint

This uses the base model as a text completer. It is not a chatbot yet; SFT and DPO come later.

In [ ]:
if RUN_STAGE == "evaluate":
    checkpoint = best if best.exists() else latest
    prompt = "Once upon a time" if MODEL == "mini_8m" else "A token is"
    run_command([
        "python", "scripts/chat.py",
        "--config", str(colab_config_path),
        "--checkpoint", str(checkpoint),
        "--prompt", prompt,
        "--max-new-tokens", "120",
    ])
else:
    print("Generation is available in the evaluate stage.")